<a href="https://colab.research.google.com/github/T-Chiunzi/Tava-Smart-Finance-Assistant/blob/main/Savings_Assistant_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pseudocode

Send greetings to the user

Display Dashboard

Ask the user to input their salary

Ask user what they want to save for and how much of their salary they want to dedicate to it

User inputs their savings goals or 0 to stop

Display to the user that they should input their expenses

Ask the user if they want to input their expenses via CSV or manually.

If via CSV, run it

If manually, ask the user to input expenses or 0 to stop

Read the data

Output table of expenses

Output a pie chart of expenses

If expenses are more than what they want to save, then advise which expenses they should reduce

Ask the user for how long they want to save for their savings goal by months or indefinitely (for emergency savings)

Ask the user if they want to increase the amount they save by (by % or by money)

Display a timeline of savings with visual of a person running along the line


In [16]:
!pip install gradio pandas hands-on-ai
import os
from getpass import getpass


os.environ['HANDS_ON_AI_SERVER'] = 'https://ollama.serveur.au'
os.environ['HANDS_ON_AI_MODEL'] = 'llama3.2'
os.environ['HANDS_ON_AI_API_KEY'] = getpass('isys2001-assignment-key')

print("🔑 Hands-on-AI server successfully configured for ISYS2001!")

isys2001-assignment-key··········
🔑 Hands-on-AI server successfully configured for ISYS2001!


In [17]:
import pandas as pd
from datetime import datetime

def calculate_savings_buddy(salary, use_secondary, secondary_name, secondary_cost, target_months, emergency_monthly, expenses_data):
    """
    Core Calculation Logic for the Savings Assistant.
    Processes salary, custom goals, and dynamically calculates expenses
    whether they are entered manually or uploaded via a CSV file.
    """
    # 1. Process Expenses (Handle both manual numbers and CSV file data)
    total_expenses = 0.0
    expense_breakdown_text = "### 📉 Monthly Expense Breakdown:\n"

    if expenses_data is not None:
        # If the user uploaded a CSV file
        if isinstance(expenses_data, str) or hasattr(expenses_data, 'name'):
            try:
                # Read the CSV (Assuming columns like 'Category' and 'Amount' or just numeric rows)
                file_path = expenses_data.name if hasattr(expenses_data, 'name') else expenses_data
                df = pd.read_csv(file_path)

                # Check standard naming or sum the last column if names differ
                if 'Amount' in df.columns:
                    total_expenses = df['Amount'].sum()
                    # Create a summary breakdown from the CSV rows
                    if 'Category' in df.columns:
                        for _, row in df.groupby('Category')['Amount'].sum().items():
                            pct = (row / salary) * 100 if salary > 0 else 0
                            expense_breakdown_text += f"- **{_}**: ${row:,.2f} ({pct:.1f}% of income)\n"
                else:
                    # Fallback to summing any numeric values if column headers are non-standard
                    total_expenses = df.select_dtypes(include=['number']).sum().sum()
                    expense_breakdown_text += f"- *Summary from uploaded document:* ${total_expenses:,.2f}\n"
            except Exception as e:
                return f"⚠️ Error processing expense CSV file: {str(e)}"
        else:
            # If the user entered expenses manually as a number in the box
            try:
                total_expenses = float(expenses_data)
                expense_breakdown_text += f"- **Manual Expenses Entry**: ${total_expenses:,.2f}\n"
            except ValueError:
                total_expenses = 0.0
                expense_breakdown_text += "- No valid expenses recorded.\n"
    else:
        expense_breakdown_text += "- No expenses recorded.\n"

    # 2. Financial Balance Calculations
    disposable_income = salary - total_expenses

    # 3. Process the Emergency Fund Requirements
    emergency_needed_monthly = float(emergency_monthly) if emergency_monthly else 0.0

    # 4. Process the Secondary Savings Goal Requirements
    secondary_needed_monthly = 0.0
    secondary_report = ""

    if use_secondary and secondary_cost and target_months:
        try:
            cost = float(secondary_cost)
            months = float(target_months)
            if months > 0:
                secondary_needed_monthly = cost / months
                secondary_report = f"- **{secondary_name or 'Secondary Goal'} Target:** Needs **${secondary_needed_monthly:,.2f}/month** over the next {months:.1f} months.\n"
        except ValueError:
            secondary_report = "⚠️ Invalid secondary goal numbers provided.\n"

    total_required_savings_monthly = emergency_needed_monthly + secondary_needed_monthly
    savings_gap = disposable_income - total_required_savings_monthly

    # 5. Build the Dynamic Evaluation Message
    if disposable_income <= 0:
        evaluation_status = "🔴 **Financial Alert:** Your expenses match or exceed your income. You do not currently have any disposable income left to save. Use the AI Chatbot tab to seek cost-reduction strategies!"
    elif savings_gap >= 0:
        evaluation_status = f"🟢 **Looking Great!** You have a disposable balance of **${disposable_income:,.2f}** remaining. After prioritizing your combined monthly goal targets (${total_required_savings_monthly:,.2f}), you still have **${savings_gap:,.2f}** left over to save or spend safely!"
    else:
        evaluation_status = f"🟡 **Budget Tightness Detected:** Your disposable income is **${disposable_income:,.2f}**, but your desired targets require **${total_required_savings_monthly:,.2f}** monthly. You are **${abs(savings_gap):,.2f} short** of hitting your timeline goals comfortably. Consider extending your goal dates or consulting the AI Advisor."

    # 6. Generate the Complete Interactive Dashboard Report
    final_markdown_report = f"""
## 📊 Your Personalized Savings Dashboard Report

**Monthly Gross Income:** ${salary:,.2f}
**Total Operational Expenses:** ${total_expenses:,.2f}
**Net Disposable Income:** ${disposable_income:,.2f}

---
{expense_breakdown_text}

---
### 🎯 Allocation Target Summary:
- **Emergency Fund Target:** Allocating **${emergency_needed_monthly:,.2f}/month**.
{secondary_report}
- **Combined Savings Target:** **${total_required_savings_monthly:,.2f}/month**

---
### 📢 Financial Health Evaluation:
{evaluation_status}
"""
    return final_markdown_report

In [18]:
# --- TEMPORARY MANUAL TEST BLOCK ---
print("🧪 Testing the new calculation function logic...")

# Call our function with dummy data to check the math engine
test_report = calculate_savings_buddy(
    salary=5000.00,
    use_secondary=True,
    secondary_name="Holiday",
    secondary_cost=1200.00,
    target_months=6,
    emergency_monthly=200.00,
    expenses_data=3000.00  # Testing with a manual number first
)

# Print the result to the console window
print(test_report)

🧪 Testing the new calculation function logic...

## 📊 Your Personalized Savings Dashboard Report

**Monthly Gross Income:** $5,000.00
**Total Operational Expenses:** $3,000.00
**Net Disposable Income:** $2,000.00

---
### 📉 Monthly Expense Breakdown:
- **Manual Expenses Entry**: $3,000.00


---
### 🎯 Allocation Target Summary:
- **Emergency Fund Target:** Allocating **$200.00/month**.
- **Holiday Target:** Needs **$200.00/month** over the next 6.0 months.

- **Combined Savings Target:** **$400.00/month**

---
### 📢 Financial Health Evaluation:
🟢 **Looking Great!** You have a disposable balance of **$2,000.00** remaining. After prioritizing your combined monthly goal targets ($400.00), you still have **$1,600.00** left over to save or spend safely!

